# Часть 2: Обучение нейросетевых моделей для классификации

В этом скрипте я обучаю модели на всех 9 подготовленных датасетах и анализирую результаты.

In [ ]:
# Проверка версии Python и установка TensorFlow
import sys
print("Python версия:", sys.version)

try:
    import tensorflow as tf
    print(f"✅ TensorFlow {tf.__version__} установлен")
except:
    print("⚠️ Устанавливаем TensorFlow...")
    !pip install tensorflow

print("\n✅ Все библиотеки готовы!")

In [ ]:
# Импортируем все необходимые библиотеки
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from google.colab import files
import io

# TensorFlow и Keras для построения нейросетей
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Sklearn для разделения данных и метрик
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

## Шаг 1: Поиск и проверка датасетов

In [ ]:
print("🔍 ПРОВЕРКА ЗАГРУЖЕННЫХ ФАЙЛОВ\n")

# Ищем датасеты в разных возможных местах
# В Colab файлы могут быть загружены в разные папки
locations = [
    '/content/datasets/',
    '/content/',
    '/content/drive/MyDrive/datasets/',  # Если используется Google Drive
]

found_files = []
dataset_path = None

# Проверяем каждую локацию
for location in locations:
    if os.path.exists(location):
        csv_files = glob.glob(os.path.join(location, '*.csv'))
        npy_x_files = glob.glob(os.path.join(location, '*_X.npy'))
        npy_y_files = glob.glob(os.path.join(location, '*_y.npy'))

        if csv_files or (npy_x_files and npy_y_files):
            dataset_path = location
            found_files = csv_files if csv_files else npy_x_files
            break

# Выводим результаты поиска
if dataset_path:
    print(f"✅ Датасеты найдены в: {dataset_path}")
    print(f"\n📁 Найдено файлов: {len(found_files)}")

    # Определяем формат
    if found_files[0].endswith('.csv'):
        print(f"📄 Формат: CSV")
        print(f"\nСписок датасетов:")
        for i, f in enumerate(sorted(found_files), 1):
            filename = os.path.basename(f)
            size_mb = os.path.getsize(f) / (1024*1024)
            print(f"  {i}. {filename} ({size_mb:.2f} MB)")
    else:
        npy_y_files = glob.glob(os.path.join(dataset_path, '*_y.npy'))
        print(f"⚡ Формат: NPY")
        print(f"   X файлов: {len(found_files)}")
        print(f"   y файлов: {len(npy_y_files)}")

    print(f"\n✅ Готово к обучению!")
else:
    print("❌ ДАТАСЕТЫ НЕ НАЙДЕНЫ!")
    print("\n📋 ИНСТРУКЦИЯ ПО ЗАГРУЗКЕ:")
    print("\n1. В левой панели Colab нажмите на значок 📁 (Files)")
    print("2. Создайте папку 'datasets' (правый клик → New folder)")
    print("3. Перетащите все NPY файлы в эту папку")
    print("4. Запустите эту ячейку снова")

In [ ]:
# Устанавливаем seed для воспроизводимости результатов
# Это важно, чтобы при повторном запуске получить те же результаты
tf.random.set_seed(42)
np.random.seed(42)

print("✅ TensorFlow версия:", tf.__version__)
print("✅ GPU доступен:", tf.config.list_physical_devices('GPU'))

## Шаг 2: Функции для построения моделей

Создаю три типа моделей:
1. **MLP (Multilayer Perceptron)** - простой перцептрон
2. **LSTM** - для работы с последовательностями
3. **RNN** - базовая рекуррентная сеть

Каждая модель имеет свои преимущества в зависимости от типа данных.

In [ ]:
def create_simple_perceptron(input_shape, num_classes):
    """
    Простой перцептрон (MLP) - многослойная полносвязная сеть.

    Когда использовать:
    - Для TF-IDF векторов (разреженные матрицы)
    - Для Word2Vec векторов (усредненные эмбеддинги)

    Преимущества:
    - Быстрое обучение
    - Хорошо работает с векторизованными признаками
    - Не требует последовательных данных

    Архитектура:
    - 3 скрытых слоя с постепенным уменьшением размерности (256->128->64)
    - Dropout для предотвращения переобучения
    - Softmax на выходе для многоклассовой классификации
    """
    model = keras.Sequential([
        layers.Input(shape=input_shape),

        # Первый слой - извлечение высокоуровневых признаков
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),  # Отключаем 30% нейронов для регуляризации

        # Второй слой - уменьшение размерности
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),

        # Третий слой - финальная обработка признаков
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),

        # Выходной слой - классификация на num_classes классов
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

def create_lstm_model(input_shape, num_classes):
    """
    LSTM (Long Short-Term Memory) - сеть с долгой краткосрочной памятью.

    Когда использовать:
    - Для частотной токенизации (последовательности индексов слов)
    - Когда важен порядок слов в тексте

    Преимущества:
    - Учитывает контекст и порядок слов
    - Может «помнить» информацию на длинных последовательностях
    - Лучше понимает смысл предложения

    Недостатки:
    - Медленнее в обучении
    - Требует больше памяти

    Архитектура:
    - 2 LSTM слоя (128 и 64 юнита)
    - return_sequences=True в первом слое для многослойной LSTM
    - Dense слой для финальной обработки
    """
    model = keras.Sequential([
        layers.Input(shape=input_shape),

        # Первый LSTM слой - обрабатывает последовательность
        # return_sequences=True передает выходы всех временных шагов следующему слою
        layers.LSTM(128, return_sequences=True),
        layers.Dropout(0.3),

        # Второй LSTM слой - дальнейшая обработка последовательности
        layers.LSTM(64),
        layers.Dropout(0.3),

        # Полносвязный слой для извлечения признаков
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),

        # Выходной слой
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

def create_rnn_model(input_shape, num_classes):
    """
    RNN (Recurrent Neural Network) - базовая рекуррентная сеть.

    Когда использовать:
    - Для экспериментов и сравнения с LSTM
    - Для коротких последовательностей

    Преимущества:
    - Быстрее LSTM
    - Проще архитектура

    Недостатки:
    - Хуже работает с длинными последовательностями
    - Проблема исчезающего градиента

    Примечание: На практике LSTM обычно дает лучшие результаты,
    но RNN полезен для понимания базовых принципов рекуррентных сетей.
    """
    model = keras.Sequential([
        layers.Input(shape=input_shape),

        # Первый RNN слой
        layers.SimpleRNN(128, return_sequences=True),
        layers.Dropout(0.3),

        # Второй RNN слой
        layers.SimpleRNN(64),
        layers.Dropout(0.3),

        # Полносвязный слой
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),

        # Выходной слой
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

print("✅ Функции моделей созданы!")
print("\nДоступные модели:")
print("  1. MLP (mlp) - простой перцептрон")
print("  2. LSTM (lstm) - рекуррентная сеть с памятью")
print("  3. RNN (rnn) - базовая рекуррентная сеть")

## Шаг 3: Конфигурация обучения

Здесь я настраиваю основные параметры обучения. Можно изменить MODEL_TYPE на 'mlp', 'lstm' или 'rnn' для экспериментов.

In [ ]:
# ОСНОВНЫЕ ПАРАМЕТРЫ ОБУЧЕНИЯ
# Можно менять для экспериментов

MODEL_TYPE = 'mlp'  # Выбираем тип модели: 'mlp', 'lstm', или 'rnn'
EPOCHS = 50  # Максимальное количество эпох
BATCH_SIZE = 32  # Размер батча (сколько примеров обрабатывается за раз)
VALIDATION_SPLIT = 0.2  # 20% от тренировочных данных для валидации
TEST_SIZE = 0.2  # 20% всех данных для финального теста
EARLY_STOPPING_PATIENCE = 5  # Остановка если нет улучшения 5 эпох

print(f"\n{'='*70}")
print("КОНФИГУРАЦИЯ ОБУЧЕНИЯ")
print(f"{'='*70}")
print(f"Модель: {MODEL_TYPE.upper()}")
print(f"Эпохи: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Размер теста: {TEST_SIZE*100}%")
print(f"Размер валидации: {VALIDATION_SPLIT*100}%")
print(f"\nПримечание: Размер теста >= 15% (требование ДЗ выполнено)")

## Шаг 4: Подготовка к обучению

In [ ]:
# Поиск и подготовка списка датасетов
print(f"\n{'='*70}")
print("ПОИСК ДАТАСЕТОВ")
print(f"{'='*70}")

# Проверяем, что датасеты найдены
if not dataset_path:
    print("❌ Датасеты не найдены! Запустите предыдущие ячейки.")
    raise FileNotFoundError("Датасеты не найдены")

# Определяем формат (CSV или NPY)
csv_files = glob.glob(os.path.join(dataset_path, '*.csv'))
csv_files = [f for f in csv_files if 'statistics' not in f.lower()]

if csv_files:
    DATASET_FORMAT = 'csv'
    dataset_files = sorted(csv_files)
    print(f"✅ Формат: CSV")
else:
    DATASET_FORMAT = 'npy'
    dataset_files = sorted(glob.glob(os.path.join(dataset_path, '*_X.npy')))
    print(f"✅ Формат: NPY")

print(f"✅ Найдено датасетов: {len(dataset_files)}")

In [ ]:
def load_dataset(file_path, format_type='npy'):
    """
    Загрузка датасета из файла.

    Поддерживает два формата:
    - NPY: отдельные файлы для X и y
    - CSV: один файл с колонкой 'label'
    """
    try:
        if format_type == 'csv':
            df = pd.read_csv(file_path)
            X = df.drop('label', axis=1).values
            y = df['label'].values
        else:  # npy формат
            X = np.load(file_path)
            y_path = file_path.replace('_X.npy', '_y.npy')
            y = np.load(y_path)

        return X.astype(np.float32), y.astype(np.int32)
    except Exception as e:
        print(f"❌ Ошибка загрузки {file_path}: {e}")
        return None, None

print("✅ Функция загрузки готова")

## Шаг 5: Главный цикл обучения

Обучаю модель на всех 9 датасетах. Для каждого датасета:
1. Загружаю данные
2. Разделяю на train и test (80/20)
3. Создаю и обучаю модель
4. Оцениваю результаты на тестовой выборке
5. Строю confusion matrix

Все результаты сохраняются для дальнейшего анализа.

In [ ]:
# Хранилища для результатов всех моделей
all_results = []
all_histories = {}
all_confusion_matrices = {}

print(f"\n{'='*70}")
print(f"НАЧАЛО ОБУЧЕНИЯ НА {len(dataset_files)} ДАТАСЕТАХ")
print(f"{'='*70}\n")

# Проходим по каждому датасету
for idx, file_path in enumerate(dataset_files, 1):
    dataset_name = os.path.basename(file_path).replace('_X.npy', '').replace('.csv', '')

    print(f"\n{'='*70}")
    print(f"ДАТАСЕТ {idx}/{len(dataset_files)}: {dataset_name}")
    print(f"{'='*70}")

    # 1. ЗАГРУЗКА ДАННЫХ
    print(f"\n[1/5] Загрузка данных...")
    X, y = load_dataset(file_path, DATASET_FORMAT)

    if X is None:
        print(f"⚠️ Пропускаем {dataset_name}")
        continue

    print(f"  ✓ Форма X: {X.shape}")
    print(f"  ✓ Форма y: {y.shape}")
    print(f"  ✓ Уникальные классы: {np.unique(y)}")
    print(f"  ✓ Распределение классов: {dict(zip(*np.unique(y, return_counts=True)))}")

    num_classes = len(np.unique(y))

    # 2. РАЗДЕЛЕНИЕ НА TRAIN И TEST
    print(f"\n[2/5] Разделение данных (train/test = {(1-TEST_SIZE)*100:.0f}/{TEST_SIZE*100:.0f})...")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=42,
        stratify=y  # Сохраняем пропорции классов
    )

    print(f"  ✓ Train: {X_train.shape[0]} примеров")
    print(f"  ✓ Test:  {X_test.shape[0]} примеров")
    print(f"  ✓ Процент тестовой выборки: {len(X_test)/len(X)*100:.1f}%")

    # 3. СОЗДАНИЕ МОДЕЛИ
    print(f"\n[3/5] Создание модели {MODEL_TYPE.upper()}...")

    # Определяем форму входных данных
    if MODEL_TYPE == 'mlp':
        # Для MLP нужна плоская форма (количество признаков)
        input_shape = (X_train.shape[1],)
        model = create_simple_perceptron(input_shape, num_classes)
    else:
        # Для LSTM/RNN нужна форма (длина_последовательности, 1)
        # Добавляем размерность для временного ряда
        X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
        X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
        input_shape = (X_train.shape[1], 1)

        if MODEL_TYPE == 'lstm':
            model = create_lstm_model(input_shape, num_classes)
        else:  # rnn
            model = create_rnn_model(input_shape, num_classes)

    # Компилируем модель
    # Adam - оптимизатор (адаптивная скорость обучения)
    # sparse_categorical_crossentropy - функция потерь для многоклассовой классификации
    # accuracy - метрика для оценки качества
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    print(f"  ✓ Модель создана")
    print(f"  ✓ Входная форма: {input_shape}")
    print(f"  ✓ Количество параметров: {model.count_params():,}")

    # 4. ОБУЧЕНИЕ МОДЕЛИ
    print(f"\n[4/5] Обучение модели...")

    # Early Stopping - останавливает обучение если нет улучшения
    # Это предотвращает переобучение и экономит время
    early_stop = EarlyStopping(
        monitor='val_loss',  # Следим за ошибкой на валидации
        patience=EARLY_STOPPING_PATIENCE,  # Ждем 5 эпох без улучшения
        restore_best_weights=True,  # Восстанавливаем лучшие веса
        verbose=1
    )

    # Обучаем модель
    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=VALIDATION_SPLIT,  # 20% train идет на валидацию
        callbacks=[early_stop],
        verbose=0  # Не выводим подробности каждой эпохи
    )

    print(f"  ✓ Обучение завершено за {len(history.history['loss'])} эпох")
    print(f"  ✓ Финальная train accuracy: {history.history['accuracy'][-1]:.4f}")
    print(f"  ✓ Финальная val accuracy: {history.history['val_accuracy'][-1]:.4f}")

    # 5. ОЦЕНКА НА ТЕСТОВОЙ ВЫБОРКЕ
    print(f"\n[5/5] Оценка на тестовой выборке...")

    # Получаем предсказания
    y_pred_proba = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)

    # Вычисляем метрики
    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

    # Отчет классификации - precision, recall, f1 для каждого класса
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

    # Total Accuracy по формуле из ДЗ: среднее precision по всем классам
    total_accuracy = np.mean([report[str(i)]['precision'] for i in range(num_classes)])

    print(f"  ✓ Test Accuracy: {test_accuracy:.4f}")
    print(f"  ✓ Test Loss: {test_loss:.4f}")
    print(f"  ✓ Total Accuracy (по формуле ДЗ): {total_accuracy:.4f}")

    # Выводим метрики по каждому классу
    print(f"\n  📊 Метрики по классам:")
    for i in range(num_classes):
        class_metrics = report[str(i)]
        print(f"     Класс {i}: Precision={class_metrics['precision']:.4f}, "
              f"Recall={class_metrics['recall']:.4f}, "
              f"F1={class_metrics['f1-score']:.4f}")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    # Сохраняем результаты
    result = {
        'dataset': dataset_name,
        'model_type': MODEL_TYPE,
        'test_accuracy': test_accuracy,
        'test_loss': test_loss,
        'total_accuracy': total_accuracy,
        'epochs_trained': len(history.history['loss']),
        'train_size': len(X_train),
        'test_size': len(X_test)
    }

    # Добавляем метрики по классам
    for i in range(num_classes):
        result[f'class_{i}_precision'] = report[str(i)]['precision']
        result[f'class_{i}_recall'] = report[str(i)]['recall']
        result[f'class_{i}_f1'] = report[str(i)]['f1-score']

    all_results.append(result)
    all_histories[dataset_name] = history.history
    all_confusion_matrices[dataset_name] = cm

    print(f"\n  ✅ Датасет {dataset_name} обработан успешно!")

print(f"\n{'='*70}")
print(f"ОБУЧЕНИЕ ЗАВЕРШЕНО!")
print(f"{'='*70}")
print(f"\nОбучено моделей: {len(all_results)}")

## Шаг 6: Анализ и визуализация результатов

In [ ]:
# Создаем сводную таблицу результатов
results_df = pd.DataFrame(all_results)

print(f"\n{'='*70}")
print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print(f"{'='*70}\n")
print(results_df[['dataset', 'test_accuracy', 'total_accuracy', 'epochs_trained']].to_string(index=False))

# Находим лучшую модель
best_idx = results_df['total_accuracy'].idxmax()
best_result = results_df.loc[best_idx]

print(f"\n{'='*70}")
print("ЛУЧШАЯ МОДЕЛЬ")
print(f"{'='*70}")
print(f"Датасет: {best_result['dataset']}")
print(f"Test Accuracy: {best_result['test_accuracy']:.4f}")
print(f"Total Accuracy: {best_result['total_accuracy']:.4f}")


In [ ]:
# Визуализация: confusion matrices для всех датасетов
print(f"\n{'='*70}")
print("ПОСТРОЕНИЕ CONFUSION MATRICES")
print(f"{'='*70}\n")

# Определяем количество строк и столбцов для subplot
n_datasets = len(all_confusion_matrices)
n_cols = 3
n_rows = (n_datasets + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
axes = axes.flatten() if n_datasets > 1 else [axes]

for idx, (dataset_name, cm) in enumerate(all_confusion_matrices.items()):
    ax = axes[idx]

    # Строим heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                cbar=False, square=True)

    ax.set_title(f'{dataset_name}\nAccuracy: {all_results[idx]["total_accuracy"]:.3f}',
                fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

# Убираем лишние subplot'ы
for idx in range(len(all_confusion_matrices), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight')
print("✅ Confusion matrices сохранены в confusion_matrices.png")
plt.show()

In [ ]:
# График сравнения accuracy по всем датасетам
plt.figure(figsize=(14, 6))

datasets_names = [r['dataset'] for r in all_results]
test_accs = [r['test_accuracy'] for r in all_results]
total_accs = [r['total_accuracy'] for r in all_results]

x = np.arange(len(datasets_names))
width = 0.35

plt.bar(x - width/2, test_accs, width, label='Test Accuracy', color='skyblue')
plt.bar(x + width/2, total_accs, width, label='Total Accuracy (ДЗ)', color='salmon')

plt.xlabel('Датасеты')
plt.ylabel('Accuracy')
plt.title(f'Сравнение результатов по всем датасетам ({MODEL_TYPE.upper()})')
plt.xticks(x, datasets_names, rotation=45, ha='right')
plt.legend()
plt.ylim(0, 1)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('accuracy_comparison.png', dpi=300, bbox_inches='tight')
print("\n✅ График сравнения сохранен в accuracy_comparison.png")
plt.show()

---

## БОНУСНОЕ ЗАДАНИЕ (+1 балл): Гибридная модель >71% точности

**Требование ДЗ:** Построить гибридную модель с Total Accuracy ≥ 71% на хотя бы одном датасете

**Что такое гибридная модель?**
Это комбинация разных архитектур, например:
- CNN + LSTM (сверточные слои для извлечения признаков + LSTM для последовательностей)
- Bidirectional LSTM (обработка последовательности в обе стороны)
- Attention механизм (фокусировка на важных словах)
- Multi-input модели (несколько типов входных данных)

### Пример: CNN + LSTM модель

In [ ]:
# Выбираю лучший датасет для гибридной модели
# Frequency токенизация лучше всего работает с LSTM
HYBRID_DATASET = 'lemmatized_frequency'

print(f"\nВыбранный датасет: {HYBRID_DATASET}")
print("Причина: Гибридные модели лучше работают с последовательностями\n")

In [ ]:
# Загрузка датасета
print("[1/6] Загрузка датасета...")

X_hybrid = np.load(f'{HYBRID_DATASET}_X.npy')
y_hybrid = np.load(f'{HYBRID_DATASET}_y.npy')

print(f"  ✓ Форма X: {X_hybrid.shape}")
print(f"  ✓ Форма y: {y_hybrid.shape}")
print(f"  ✓ Количество классов: {len(np.unique(y_hybrid))}")

In [ ]:
# Разделение на train/test
print("\n[2/6] Разделение на train/test...")

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_hybrid, y_hybrid,
    test_size=0.2,
    random_state=42,
    stratify=y_hybrid
)

print(f"  ✓ Train: {X_train_h.shape}")
print(f"  ✓ Test: {X_test_h.shape}")

In [ ]:
# Создание гибридной модели CNN + LSTM
print("\n[3/6] Создание гибридной модели...")

def create_hybrid_model(input_shape, num_classes):
    """
    ГИБРИДНАЯ АРХИТЕКТУРА: CNN + LSTM

    Почему это работает лучше:
    1. Embedding - преобразует индексы слов в плотные векторы
    2. Conv1D слои - находят локальные паттерны (фразы типа "very good")
    3. MaxPooling - уменьшает размерность, оставляя важное
    4. LSTM - понимает порядок слов и долгосрочные зависимости
    5. Dropout - предотвращает переобучение
    6. Dense - итоговая классификация
    """
    model = keras.Sequential([
        # Входной слой
        layers.Input(shape=input_shape),

        # Embedding: преобразуем индексы слов в векторы размерности 128
        # input_dim=2000 - размер словаря (максимальный индекс слова)
        layers.Embedding(input_dim=2000, output_dim=128),

        # Первый свёрточный блок
        # Conv1D с 64 фильтрами и окном размера 3
        # Находит локальные паттерны из 3 последовательных слов
        layers.Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.3),

        # Второй свёрточный блок
        # Conv1D с 32 фильтрами - более абстрактные паттерны
        layers.Conv1D(filters=32, kernel_size=3, activation='relu', padding='same'),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.3),

        # LSTM слой для понимания последовательности
        # 64 units - размер скрытого состояния
        # return_sequences=False - возвращаем только последний выход
        layers.LSTM(64, return_sequences=False),
        layers.Dropout(0.5),

        # Dense слои для классификации
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),

        # Выходной слой с softmax для вероятностей классов
        layers.Dense(num_classes, activation='softmax')
    ])

    return model

# Создаём модель
num_classes_h = len(np.unique(y_hybrid))
input_shape_h = (X_hybrid.shape[1],)

model_hybrid = create_hybrid_model(input_shape_h, num_classes_h)

# Компилируем модель
model_hybrid.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),  # Меньший learning rate
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n✓ Гибридная модель создана!")
print("\nАрхитектура модели:")
model_hybrid.summary()

In [ ]:
# Настройка callbacks для улучшения обучения
print("\n[4/6] Настройка callbacks...")

# Early Stopping - останавливает обучение если нет улучшения
early_stop_hybrid = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,  # Больше терпения для гибридной модели
    restore_best_weights=True,
    verbose=1
)

# ReduceLROnPlateau - уменьшает learning rate при застревании
reduce_lr_hybrid = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,  # Уменьшаем LR в 2 раза
    patience=5,
    min_lr=0.00001,
    verbose=1
)

print("  ✓ Early Stopping (patience=15)")
print("  ✓ ReduceLROnPlateau (уменьшение learning rate)")

In [ ]:
# Обучение модели
print("\n[5/6] Обучение гибридной модели...")
print("Это может занять несколько минут...\n")

history_hybrid = model_hybrid.fit(
    X_train_h, y_train_h,
    validation_split=0.2,
    epochs=100,  # Максимум эпох
    batch_size=32,
    callbacks=[early_stop_hybrid, reduce_lr_hybrid],
    verbose=1
)

print("\n✓ Обучение завершено!")

In [ ]:
# Оценка модели
print("\n[6/6] Оценка модели на тестовых данных...\n")

# Предсказания
y_pred_hybrid = model_hybrid.predict(X_test_h, verbose=0)
y_pred_classes_hybrid = np.argmax(y_pred_hybrid, axis=1)

# Метрики
test_loss_hybrid, test_acc_hybrid = model_hybrid.evaluate(X_test_h, y_test_h, verbose=0)

print("="*70)
print("РЕЗУЛЬТАТЫ ГИБРИДНОЙ МОДЕЛИ")
print("="*70)
print(f"\nДатасет: {HYBRID_DATASET}")
print(f"Архитектура: CNN + LSTM (Гибридная)")
print(f"Test Loss: {test_loss_hybrid:.4f}")
print(f"Test Accuracy: {test_acc_hybrid:.4f} ({test_acc_hybrid*100:.2f}%)")

# Classification report
print("\nDetailed Classification Report:")
print(classification_report(y_test_h, y_pred_classes_hybrid,
                          target_names=[f'Class {i}' for i in range(num_classes_h)]))

# Total Accuracy по формуле из ДЗ
report_dict_hybrid = classification_report(y_test_h, y_pred_classes_hybrid,
                                           output_dict=True, zero_division=0)
precisions_hybrid = [report_dict_hybrid[str(i)]['precision']
                     for i in range(num_classes_h)]
total_accuracy_hybrid = np.mean(precisions_hybrid)

print(f"\n{'='*70}")
print(f"📊 TOTAL ACCURACY (по формуле ДЗ): {total_accuracy_hybrid:.4f} ({total_accuracy_hybrid*100:.2f}%)")
print(f"{'='*70}")

In [ ]:
# Confusion Matrix
print("\nПостроение Confusion Matrix...")

cm_hybrid = confusion_matrix(y_test_h, y_pred_classes_hybrid)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_hybrid, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'Class {i}' for i in range(num_classes_h)],
            yticklabels=[f'Class {i}' for i in range(num_classes_h)])
plt.title(f'Confusion Matrix: HYBRID MODEL ({HYBRID_DATASET})')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(f'confusion_matrix_hybrid_{HYBRID_DATASET}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Сохранено: confusion_matrix_hybrid_{HYBRID_DATASET}.png")

In [ ]:
# Графики обучения
print("\nПостроение графиков обучения...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График точности
axes[0].plot(history_hybrid.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history_hybrid.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].set_title('Model Accuracy (Hybrid CNN+LSTM)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# График функции потерь
axes[1].plot(history_hybrid.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history_hybrid.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_title('Model Loss (Hybrid CNN+LSTM)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'training_history_hybrid_{HYBRID_DATASET}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Сохранено: training_history_hybrid_{HYBRID_DATASET}.png")

## Итоги второй части:

✅ **Выполнено:**
1. Созданы функции для 3 типов моделей (MLP, LSTM, RNN)
2. Обучены модели на всех 9 датасетах
3. Соблюдено требование: test size >= 15%
4. Для каждой модели рассчитаны:
   - Precision, Recall, F1-score по каждому классу
   - Test Accuracy
   - Total Accuracy (по формуле ДЗ)
5. Построены Confusion Matrices для всех 9 датасетов
6. Создана сводная таблица результатов
7. Определена лучшая модель
